# Kiểm tra hệ thống RAG
Notebook này kiểm tra luồng chạy của hệ thống, thời gian thực thi của từng chức năng, số lượng token tiêu thụ và đánh giá chức năng (pass or not).
Có 2 chế độ: **Bật Think** và **Không bật Think**.

In [4]:
import time
import json
from typing import Any, Dict
from langchain_core.callbacks import BaseCallbackHandler

from llm_handler import app_graph

class TokenCallbackHandler(BaseCallbackHandler):
    """Callback để theo dõi số lượng token sử dụng."""
    def __init__(self):
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.total_tokens = 0

    def on_llm_end(self, response: Any, **kwargs: Any) -> None:
        try:
            if response.generations:
                gen = response.generations[0][0]
                if hasattr(gen, 'message') and hasattr(gen.message, 'response_metadata'):
                    usage = gen.message.response_metadata.get('token_usage', {})
                    if usage:
                        self.prompt_tokens += usage.get('prompt_tokens', 0)
                        self.completion_tokens += usage.get('completion_tokens', 0)
                        self.total_tokens += usage.get('total_tokens', 0)
        except Exception as e:
            pass
            
    def reset(self):
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.total_tokens = 0

token_tracker = TokenCallbackHandler()

c:\Users\dongg\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Reranker Model...
Reranker Model Loaded.


In [5]:
def run_test(query: str, use_think: bool):
    print(f"\n{'='*50}")
    print(f"BẮT ĐẦU TEST - CHẾ ĐỘ THINK = {use_think}")
    print(f"Câu hỏi: '{query}'")
    print(f"{'='*50}\n")
    
    inputs = {"user_query": query, "use_think": use_think}
    token_tracker.reset()
    
    start_time = time.time()
    last_time = start_time
    
    passed = True
    
    try:
        # app_graph.stream sẽ chạy từng chức năng (node) và trả kết quả liên tục
        for output in app_graph.stream(inputs, config={"callbacks": [token_tracker]}):
            current_time = time.time()
            for node_name, node_state in output.items():
                elapsed = current_time - last_time
                print(f"▶ Chức năng [{node_name}] hoạt động ổn. Thời gian chạy: {elapsed:.2f} giây")
                
                # In ra một số kết quả tóm tắt của node để kiểm tra
                if node_name == 'extract_metadata_hyde':
                    print(f"  - Metadata trích xuất: {node_state.get('extracted_metadata')}")
                elif node_name in ['retrieve_and_process_advanced', 'retrieve_basic']:
                    chunks = node_state.get('final_chunks', [])
                    print(f"  - Số lượng tài liệu tìm thấy: {len(chunks)}")
                elif node_name == 'generate_response':
                    resp = node_state.get('final_response', '')
                    print(f"  - Độ dài câu trả lời: {len(resp)} ký tự")
                
            last_time = current_time
            
        total_time = time.time() - start_time
        print(f"\n{'-'*50}")
        print(f"TỔNG THỜI GIAN CHẠY: {total_time:.2f} giây")
        print(f"TỔNG TOKEN TIÊU THỤ: {token_tracker.total_tokens} (Prompt: {token_tracker.prompt_tokens}, Completion: {token_tracker.completion_tokens})")
        print(f"KẾT QUẢ: {'✅ PASS' if passed else '❌ NOT PASS'}")
        print(f"{'-'*50}")
        
    except Exception as e:
        passed = False
        total_time = time.time() - start_time
        print(f"\n❌ LỖI XẢY RA TRONG QUÁ TRÌNH CHẠY: {str(e)}")
        print(f"{'-'*50}")
        print(f"TỔNG THỜI GIAN ĐẾN KHI LỖI: {total_time:.2f} giây")
        print(f"KẾT QUẢ: ❌ NOT PASS")
        print(f"{'-'*50}")

### Khai báo câu hỏi đầu vào (Biến nhập text)

In [6]:
# Có thể thay đổi đoạn text (câu hỏi) cần chạy tại biến này
test_query = "Tóm tắt nội dung của Điều 10 Thông tư 09/2016/TT-BXD hướng dẫn hợp đồng thi công xây dựng công trình"

### 1. Phiên bản KHÔNG bật Think (`use_think = False`)

In [7]:
run_test(query=test_query, use_think=False)


BẮT ĐẦU TEST - CHẾ ĐỘ THINK = False
Câu hỏi: 'Tóm tắt nội dung của Điều 10 Thông tư 09/2016/TT-BXD hướng dẫn hợp đồng thi công xây dựng công trình'

▶ Chức năng [retrieve_basic] hoạt động ổn. Thời gian chạy: 0.76 giây
  - Số lượng tài liệu tìm thấy: 3
▶ Chức năng [generate_response] hoạt động ổn. Thời gian chạy: 1.20 giây
  - Độ dài câu trả lời: 101 ký tự

--------------------------------------------------
TỔNG THỜI GIAN CHẠY: 1.96 giây
TỔNG TOKEN TIÊU THỤ: 1032 (Prompt: 670, Completion: 362)
KẾT QUẢ: ✅ PASS
--------------------------------------------------


### 2. Phiên bản BẬT Think (`use_think = True`)

In [8]:
run_test(query=test_query, use_think=True)


BẮT ĐẦU TEST - CHẾ ĐỘ THINK = True
Câu hỏi: 'Tóm tắt nội dung của Điều 10 Thông tư 09/2016/TT-BXD hướng dẫn hợp đồng thi công xây dựng công trình'

[DEBUG] Raw LLM response for metadata extraction (cleaned): {
    "extracted_metadata": {
        "issuing_agency": "Bộ Xây dựng",
        "promulgation_date": "",
        "sign_number": "09/2016/TT-BXD",
        "signer": "",
        "type": "Thông tư"
    },
    "hypothetical_doc": "Điều 10 Thông tư 09/2016/TT-BXD hướng dẫn về hợp đồng thi công xây dựng công trình, quy định các nguyên tắc cơ bản trong việc lập, thực hiện và quản lý hợp đồng thi công. Nội dung tập trung vào quyền và nghĩa vụ của các bên, điều kiện nghiệm thu, thanh toán và xử lý tranh chấp phát sinh trong quá trình thi công."
}
▶ Chức năng [extract_metadata_hyde] hoạt động ổn. Thời gian chạy: 2.17 giây
  - Metadata trích xuất: {'issuing_agency': 'Bộ Xây dựng', 'promulgation_date': '', 'sign_number': '09/2016/TT-BXD', 'signer': '', 'type': 'Thông tư'}
[DEBUG] Passed chunks